# DGD-TabPA — Full Experiments (Google Colab)

Run the complete evaluation pipeline on Colab (GPU recommended).

**Runtime → Change runtime type → GPU (T4 / A100)**

### What this notebook does
1. Clone / mount the project  
2. Install dependencies  
3. Download benchmark datasets  
4. Run DGD end-to-end + SMOTE baseline  
5. Optionally run multi-dataset / ablation / privacy suites  
6. Collect results and download a zip of outputs  

Outputs are written under `DGD-TabPA/outputs/experiments/`.

## 0. Configuration

Edit these cells before running the rest.

In [ ]:
# --- Project source ---
# Option A (default): clone from GitHub
REPO_URL = "https://github.com/suranjaliyanage/DGD-TabPA.git"
REPO_BRANCH = "master"
# If the repo is private, set a GitHub PAT (keep secret; do not commit):
GITHUB_TOKEN = ""  # e.g. "ghp_..."

# Option B: use a zip already uploaded to Colab, or a Drive folder
USE_DRIVE = False
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/DGD-TabPA"  # only if USE_DRIVE=True

# --- Experiment knobs ---
PRIMARY_DATASET = "diabetes"
EPOCHS = 30                 # draft: 30–50 | stronger: 100+
DISTILL_EPOCHS = 50         # draft: 50 | stronger: 100+
BATCH_SUITE_EPOCHS = 30
RUN_HEAVY = False           # True = core_fast + ablations + privacy + smote suite
DOWNLOAD_DATASETS = "all"   # "all" or a single key e.g. "diabetes"

print("Config ready.")

## 1. Environment check (GPU)

In [ ]:
import sys
import os
import shutil
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Colab: {IN_COLAB}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Enable Runtime → Change runtime type → GPU.")

## 2. Get the project code

Clones into `/content/DGD-TabPA` (or mounts Drive if `USE_DRIVE=True`).

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path(DRIVE_PROJECT_PATH)
    if not PROJECT_ROOT.exists():
        raise FileNotFoundError(
            f"Drive path not found: {PROJECT_ROOT}\n"
            "Upload/unzip the project there, or set USE_DRIVE=False to clone from GitHub."
        )
else:
    PROJECT_ROOT = Path("/content/DGD-TabPA")
    if PROJECT_ROOT.exists():
        print(f"Removing existing clone at {PROJECT_ROOT}")
        shutil.rmtree(PROJECT_ROOT)

    clone_url = REPO_URL
    if GITHUB_TOKEN.strip():
        # https://<token>@github.com/owner/repo.git
        clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN.strip()}@")

    cmd = ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", clone_url, str(PROJECT_ROOT)]
    print(" ".join(cmd).replace(GITHUB_TOKEN, "***") if GITHUB_TOKEN else " ".join(cmd))
    subprocess.check_call(cmd)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")
print("Top-level:", sorted(p.name for p in PROJECT_ROOT.iterdir())[:20])

## 3. Install dependencies

Colab already has PyTorch; we install the remaining project requirements.

In [ ]:
!pip -q install -r requirements.txt

# Quick import sanity check
import yaml
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown, FileLink

print("Dependencies OK")

In [ ]:
EXPERIMENTS_DIR = PROJECT_ROOT / "outputs" / "experiments"
MASTER_CSV = EXPERIMENTS_DIR / "results_master.csv"
CONFIG = "config/default.yaml"

def run_cmd(args, check=True):
    cmd = [sys.executable, *args]
    print("\n>>>", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result.returncode

def show_figures(run_id, max_figs=8):
    fig_dir = EXPERIMENTS_DIR / run_id / "figures"
    if not fig_dir.exists():
        print(f"No figures at {fig_dir}")
        return
    for p in sorted(fig_dir.glob("*.png"))[:max_figs]:
        display(Markdown(f"**{p.name}**"))
        display(Image(filename=str(p)))

def load_summary(run_id):
    path = EXPERIMENTS_DIR / run_id / "summary_row.json"
    if not path.exists():
        print(f"Missing {path}")
        return None
    import json
    with open(path, encoding="utf-8") as f:
        return json.load(f)

print("Helpers ready.")

## 4. Download datasets

In [ ]:
if DOWNLOAD_DATASETS == "all":
    run_cmd(["scripts/download_data.py"])
else:
    run_cmd(["scripts/download_data.py", "--dataset", DOWNLOAD_DATASETS])

raw = list((PROJECT_ROOT / "data" / "raw").glob("*.csv"))
print(f"Downloaded {len(raw)} CSVs:")
for p in raw:
    print(" -", p.name)

## 5. Single end-to-end DGD run

Train → distill → evaluate → figures for `PRIMARY_DATASET`.

In [ ]:
RUN_ID = f"{PRIMARY_DATASET}_dgd"

run_cmd([
    "scripts/run_experiment.py",
    "--config", CONFIG,
    "--dataset", PRIMARY_DATASET,
    "--method", "dgd_tabpa",
    "--epochs", str(EPOCHS),
    "--distill-epochs", str(DISTILL_EPOCHS),
    "--run-id", RUN_ID,
])

summary = load_summary(RUN_ID)
display(summary)
show_figures(RUN_ID)

## 6. SMOTE baseline (same evaluation protocol)

In [ ]:
SMOTE_ID = f"{PRIMARY_DATASET}_smote"

run_cmd([
    "scripts/run_experiment.py",
    "--config", CONFIG,
    "--dataset", PRIMARY_DATASET,
    "--method", "smote",
    "--run-id", SMOTE_ID,
])

display(pd.DataFrame([load_summary(RUN_ID), load_summary(SMOTE_ID)]))

## 7. Heavy suites (optional)

Set `RUN_HEAVY = True` in the config cell, then re-run this section.

- `core_fast` — multi-dataset quantitative table  
- `ablations` — mlp / no_attention / minmax / raw_space  
- `privacy` — ε sweep + trade-off plot  
- `smote` — SMOTE on multiple datasets

In [ ]:
if RUN_HEAVY:
    run_cmd([
        "scripts/run_batch_experiments.py",
        "--suite", "core_fast",
        "--epochs", str(BATCH_SUITE_EPOCHS),
    ])
    run_cmd([
        "scripts/run_batch_experiments.py",
        "--suite", "ablations",
        "--dataset", PRIMARY_DATASET,
        "--epochs", str(BATCH_SUITE_EPOCHS),
    ])
    run_cmd([
        "scripts/run_batch_experiments.py",
        "--suite", "privacy",
        "--dataset", PRIMARY_DATASET,
        "--epochs", str(BATCH_SUITE_EPOCHS),
    ])
    run_cmd(["scripts/run_batch_experiments.py", "--suite", "smote"])

    tradeoff = EXPERIMENTS_DIR / f"privacy_utility_{PRIMARY_DATASET}.png"
    if tradeoff.exists():
        display(Image(filename=str(tradeoff)))
else:
    print("Skipped heavy suites (set RUN_HEAVY=True to enable).")
    print("CLI equivalents:")
    print(f"  python scripts/run_batch_experiments.py --suite core_fast --epochs {BATCH_SUITE_EPOCHS}")
    print(f"  python scripts/run_batch_experiments.py --suite ablations --dataset {PRIMARY_DATASET} --epochs {BATCH_SUITE_EPOCHS}")
    print(f"  python scripts/run_batch_experiments.py --suite privacy --dataset {PRIMARY_DATASET} --epochs {BATCH_SUITE_EPOCHS}")
    print("  python scripts/run_batch_experiments.py --suite smote")

## 8. Collect results table

In [ ]:
import json

if MASTER_CSV.exists():
    master = pd.read_csv(MASTER_CSV)
    print(f"Rows in master table: {len(master)}")
    display(master)

    cols = [
        c for c in [
            "dataset", "method",
            "wasserstein_mean", "pcd", "jsd_mean",
            "mean_tstr_f1", "mean_f1_gap", "mean_tstr_auc",
            "dcr_median", "dcr_5th_percentile", "mia_auc",
            "privacy_rating", "epsilon",
        ] if c in master.columns
    ]
    compact = master[cols].round(4)
    display(compact)

    out_table = EXPERIMENTS_DIR / "results_table.csv"
    compact.to_csv(out_table, index=False)
    print(f"Saved -> {out_table}")
else:
    print(f"No master CSV yet at {MASTER_CSV}")

fig_files = sorted(EXPERIMENTS_DIR.glob("**/figures/*.png"))
print(f"\nTotal figures: {len(fig_files)}")
for p in fig_files[:30]:
    print(" -", p.relative_to(PROJECT_ROOT))

## 9. Zip & download outputs

Downloads `dgd_tabpa_outputs.zip` to your local machine.

In [ ]:
from datetime import datetime

stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
zip_base = Path("/content") / f"dgd_tabpa_outputs_{stamp}"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=str(EXPERIMENTS_DIR)))
print(f"Created: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print("Not in Colab — zip saved at:", zip_path)

## 10. Optional — copy outputs to Google Drive

Useful if the Colab session may disconnect.

In [ ]:
SAVE_TO_DRIVE = False  # set True to copy
DRIVE_OUT = "/content/drive/MyDrive/DGD-TabPA-outputs"

if SAVE_TO_DRIVE:
    from datetime import datetime
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    dest = Path(DRIVE_OUT)
    dest.mkdir(parents=True, exist_ok=True)
    stamp_local = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    target = dest / f"experiments_{stamp_local}"
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(EXPERIMENTS_DIR, target)
    print(f"Copied experiments -> {target}")
else:
    print("Skipped Drive copy (set SAVE_TO_DRIVE=True to enable).")

---
## Checklist

| Step | Status when done |
|---|---|
| GPU enabled | `CUDA available: True` |
| Project cloned / mounted | `Project root: /content/DGD-TabPA` |
| Data downloaded | CSVs listed under `data/raw/` |
| DGD run finished | `summary_row.json` + figures |
| SMOTE baseline finished | comparison table shown |
| Outputs downloaded | `dgd_tabpa_outputs_*.zip` |

For stronger numbers, raise `EPOCHS` / `DISTILL_EPOCHS` and set `RUN_HEAVY = True`.